# Imports


In [ ]:
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

import mascope_sdk
from mascope_file.name import get_instrument_type
from mascope_signal.instrument_func.fit import (
    get_peak_shape,  # TODO fix this broken import
    get_resolution_function,
    process_peak_shapes,
    r_tof,
)
from mascope_signal.peak import fit_n_peaks, segment_spec

# Pick the sample file


In [ ]:
mascope_url = "https://192.168.1.99/"

# Show list of workspaces
workspace_df = pd.DataFrame(mascope_sdk.get_workspaces(mascope_url))
workspace_df

In [ ]:
# Show batches in a workspace
workspace_id = "XSG5SI9bNKVwF7P7"

batch_df = pd.DataFrame(
    mascope_sdk.get_sample_batches(mascope_url, workspace_id=workspace_id)
)
batch_df

In [ ]:
# Show samples in a batch
batch_id = "0OIf8DaWYcEkjwPh"

samples = mascope_sdk.get_sample_batch_data(mascope_url, sample_batch_id=batch_id)
samples_df = pd.DataFrame(samples["samples"])
query = "sample_item_name.str.contains('100ng')"
samples_df.query(query)

In [ ]:
# Show chosen sample info
sample_loc = 10

sample_series = samples_df.iloc[sample_loc]
sample_series

In [ ]:
file_id = sample_series.sample_file_id
filename = sample_series.filename

# Load peaks and spectrum
sample_peaks = mascope_sdk.get_sample_file_peaks(mascope_url, file_id, areas=False)
sample_spectrum = mascope_sdk.get_sample_file_spectrum(mascope_url, file_id)

# Load spectrum of the first file in the batch
first_sample_spectrum = mascope_sdk.get_sample_file_spectrum(
    mascope_url, samples_df.iloc[0].sample_file_id
)
first_sample_name = samples_df.iloc[0].filename

# Calculate instrument functions


In [ ]:
def calculate_instrument_function(mz, spec, filename):
    """The function calculates instrument function based of the spectrum and filename"""
    # Define the MS
    instrument_type = get_instrument_type(filename)

    # Make sure numpy arrays are provided
    mz = np.array(mz)
    spec = np.array(spec)

    # Get x-domain, normalized peak shapes and associated peak positions and FWHMs
    p_x, p_ys, p_mzs, p_fwhms = process_peak_shapes(
        mz, spec, instrument_type, dmz=0.5, r_sq_thres=0.95
    )

    # Convert values to numpy arrays
    p_mzs = np.array(p_mzs)
    p_fwhms = np.array(p_fwhms)

    # Calculate instrument functions
    ps = get_peak_shape(p_x, p_ys)
    res_fun = get_resolution_function(instrument_type, p_mzs, p_fwhms, ndev=1)

    return {"peakshape": ps, "resolution_function": res_fun}

In [ ]:
# Get instrument function in a current way
request_body = f"{mascope_url}api/instrument_functions/?filename={filename}".replace(
    "+", "%2B"
)
message = requests.get(request_body, verify=False).json()
old_inst_fun = {
    "peakshape": message["peakshape"],
    "resolution_function": partial(
        r_tof, a=message["resolution_function"][0], b=message["resolution_function"][1]
    ),
    # "resolution_function": np.poly1d(message["resolution_function"])
}

# For Orbitrap use this
old_inst_fun["resolution_function"] = lambda mz: 1693201.8660751865 / np.sqrt(mz)

# Get individual instrument function
ind_inst_fun = calculate_instrument_function(
    sample_spectrum["mz"], sample_spectrum["intensity"], filename
)

# Get batch instrument function
batch_inst_fun = calculate_instrument_function(
    first_sample_spectrum["mz"], first_sample_spectrum["intensity"], first_sample_name
)

# Fit spectra with different instrument functions


In [ ]:
def segment_array(instrument_type, mz, spec, dmz, min_mz, max_mz):
    """Segment either spectrum or fitted peaks for comparison or peak fitting"""
    mz = np.array(mz)
    spec = np.array(spec)
    if instrument_type == "tof":
        u_list = np.arange(min_mz, max_mz + 1)
        segmented_indices = [
            np.where(np.logical_and(mz >= u - dmz, mz <= u + dmz)) for u in u_list
        ]
    if instrument_type == "orbi":
        segmented_indices = segment_spec(spec)
    else:
        segmented_indices = []
        temp_mz = min_mz
        while temp_mz < max_mz:
            segmented_indices.append(
                np.where(np.logical_and(mz >= temp_mz, mz <= temp_mz + dmz))
            )
            temp_mz += dmz

    return [(mz[chunk], spec[chunk]) for chunk in segmented_indices]

In [ ]:
def fit_spectrum(instrument_type, mz, spec, dmz, peak_shape, res_fun, threshold=0.9):
    """Apply common Mascope peak fitting algorithm to the spectrum"""
    mz = np.array(mz)
    spec = np.array(spec)

    specs_to_fit = segment_array(instrument_type, mz, spec, dmz, min(mz), max(mz))

    fitted_peaks = []
    for mz_chunk, spec_chunk in specs_to_fit:
        fit, fitted_chunk = fit_n_peaks(
            mz_chunk,
            spec_chunk,
            peak_shape,
            res_fun,
            threshold,
        )
        if fit is None:
            # Nothing to fit
            continue

        for i in fitted_chunk:
            fitted_peaks.append(i)

    # Convert fitted peaks to numpy array
    fitted_peaks = np.asarray(fitted_peaks)
    # Get fitted peak positions and heights
    fitted_pos = fitted_peaks[:, 0]
    fitted_hei = fitted_peaks[:, 1]

    return {"mzs": fitted_pos, "heis": fitted_hei}

In [ ]:
instrument_type = get_instrument_type(filename)

init_peaks = fit_spectrum(
    instrument_type,
    sample_spectrum["mz"],
    sample_spectrum["intensity"],
    dmz=0.5,
    peak_shape=old_inst_fun["peakshape"],
    res_fun=old_inst_fun["resolution_function"],
)

indi_peaks = fit_spectrum(
    instrument_type,
    sample_spectrum["mz"],
    sample_spectrum["intensity"],
    dmz=0.5,
    peak_shape=ind_inst_fun["peakshape"],
    res_fun=ind_inst_fun["resolution_function"],
)

batch_peaks = fit_spectrum(
    instrument_type,
    sample_spectrum["mz"],
    sample_spectrum["intensity"],
    dmz=0.5,
    peak_shape=batch_inst_fun["peakshape"],
    res_fun=batch_inst_fun["resolution_function"],
)

# Compare results for different instrument functions


In [ ]:
print(
    f"""Number of detected peaks for instrument functions:






    - Retrieved by time (current approach): {len(init_peaks["mzs"])}






    - Individually culaculated: {len(indi_peaks["mzs"])}






    - One per batch: {len(batch_peaks["mzs"])}
"""
)

In [ ]:
# Segment results to compare chunk by chunk
min_mz = np.min(sample_spectrum["mz"])
max_mz = np.max(sample_spectrum["mz"])

# Change mz_step to 0.5 for TOF and pass proper instrument_type
mz_step = 0.1

spec_segmented = segment_array(
    None, sample_spectrum["mz"], sample_spectrum["intensity"], mz_step, min_mz, max_mz
)

init_segmented = segment_array(
    None, init_peaks["mzs"], init_peaks["heis"], mz_step, min_mz, max_mz
)
indi_segmented = segment_array(
    None, indi_peaks["mzs"], indi_peaks["heis"], mz_step, min_mz, max_mz
)
batch_segmented = segment_array(
    None, batch_peaks["mzs"], batch_peaks["heis"], mz_step, min_mz, max_mz
)

## Plot and save figures

We consider chunks with a difference in a number of fitted peaks


In [ ]:
# Compare batch and individual
for i in range(len(indi_segmented)):
    # Check if number of peak per segment is the same
    if len(indi_segmented[i][0]) != len(batch_segmented[i][0]):
        plt.figure(figsize=(10, 2))
        plt.plot(
            spec_segmented[i][0],
            spec_segmented[i][1],
            label="Raw signal",
            color="black",
            linewidth=3,
        )
        plt.vlines(
            x=batch_segmented[i][0],
            ymin=0,
            ymax=batch_segmented[i][1],
            label="Batch",
            color="red",
            linewidth=3,
        )
        plt.vlines(
            x=indi_segmented[i][0],
            ymin=0,
            ymax=indi_segmented[i][1],
            label="Individual",
            color="blue",
        )
        plt.xlabel("M/z [Th]")
        plt.ylabel("Counts [a.u.]")

        x_min = np.min([np.min(indi_segmented[i][0]), np.min(batch_segmented[i][0])])
        x_max = np.max([np.max(indi_segmented[i][0]), np.max(batch_segmented[i][0])])

        # plt.xlim(x_min - 0.1, x_max + 0.1)
        plt.legend()
        plt.savefig(f"C:/Users/alvai/Desktop/temp/figs/{i}.png")
        plt.close()

In [ ]:
# Compare batch and old way of calculating instrument functions
for i in range(len(init_segmented)):
    # Check if number of peak per segment is the same
    if len(init_segmented[i][0]) != len(batch_segmented[i][0]):
        plt.figure(figsize=(10, 2))
        plt.plot(
            spec_segmented[i][0],
            spec_segmented[i][1],
            label="Raw signal",
            color="black",
            linewidth=3,
        )
        plt.vlines(
            x=batch_segmented[i][0],
            ymin=0,
            ymax=batch_segmented[i][1],
            label="Batch",
            color="red",
            linewidth=3,
        )
        plt.vlines(
            x=init_segmented[i][0],
            ymin=0,
            ymax=init_segmented[i][1],
            label="By time",
            color="blue",
        )
        plt.xlabel("M/z [Th]")
        plt.ylabel("Counts [a.u.]")

        try:
            x_min = np.min(
                [np.min(init_segmented[i][0]), np.min(batch_segmented[i][0])]
            )
            x_max = np.max(
                [np.max(init_segmented[i][0]), np.max(batch_segmented[i][0])]
            )
        except ValueError:
            plt.close()
            continue

        plt.xlim(x_min - 0.1, x_max + 0.1)
        plt.legend()
        plt.savefig(f"C:/Users/alvai/Desktop/temp/figs/{i}.png")
        plt.close()

In [ ]:
# Compare individual and by time
for i in range(len(indi_segmented)):
    # Check if number of peak per segment is the same
    if init_segmented[i][0].size != indi_segmented[i][0].size:
        plt.figure(figsize=(10, 2))
        plt.plot(
            spec_segmented[i][0],
            spec_segmented[i][1],
            label="Raw signal",
            color="black",
            linewidth=3,
        )
        plt.vlines(
            x=init_segmented[i][0],
            ymin=0,
            ymax=init_segmented[i][1],
            label="By time",
            color="red",
            linewidth=3,
        )
        plt.vlines(
            x=indi_segmented[i][0],
            ymin=0,
            ymax=indi_segmented[i][1],
            label="Per sample",
            color="blue",
        )
        plt.xlabel("M/z [Th]")
        plt.ylabel("Counts [a.u.]")

        try:
            x_min = np.min([np.min(init_segmented[i][0]), np.min(indi_segmented[i][0])])
            x_max = np.max([np.max(init_segmented[i][0]), np.max(indi_segmented[i][0])])
        except ValueError as e:
            print(e)
            plt.close()
            continue

        # plt.xlim(x_min - 0.1, x_max + 0.1)
        plt.legend()
        plt.savefig(f"C:/Users/alvai/Desktop/temp/figs/{i}.png")
        plt.close()